In [1]:
import cv2
import datetime
import threading



In [2]:
import time

cap = cv2.VideoCapture('http://planktoscope.local/ps/hal/camera/streams/preview.mjpg')

frame_count = 0
start_time = time.time()

while frame_count < 100:
    ret, frame = cap.read()
    if not ret:
        break
    frame_count += 1

end_time = time.time()
duration = end_time - start_time
fps = frame_count / duration
print(f"Estimated FPS: {fps:.2f}")

cap.release()


Estimated FPS: 10.11


In [3]:
stream_url = 'http://planktoscope.local/ps/hal/camera/streams/preview.mjpg'
cap = cv2.VideoCapture(stream_url)

if not cap.isOpened():
    print("Error: Cannot open video stream.")
    exit()

recording = False
out = None

def input_listener():
    global recording, out
    while True:
        cmd = input("Type 'start' to begin recording, 'stop' to stop, 'quit' to exit:\n").strip().lower()
        if cmd == 'start' and not recording:
            timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
            filename = f"./samples/microscopy_{timestamp}.mp4"
            fourcc = cv2.VideoWriter_fourcc(*'mp4v')
            fps = 10.0  # You can measure this as explained earlier
            frame_size = (int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)))
            out = cv2.VideoWriter(filename, fourcc, fps, frame_size)
            recording = True
            print(f"[+] Started recording to {filename}")
        elif cmd == 'stop' and recording:
            recording = False
            out.release()
            out = None
            print("[*] Stopped recording.")
        elif cmd == 'quit':
            recording = False
            if out:
                out.release()
            cap.release()
            print("[x] Exiting.")
            exit()

# Start the input listener thread
input_thread = threading.Thread(target=input_listener, daemon=True)
input_thread.start()

# Main loop to read and optionally record
while True:
    ret, frame = cap.read()
    if not ret:
        print("[-] Frame read failed.")
        break
    if recording and out:
        out.write(frame)


[+] Started recording to ./samples/microscopy_20250723_131605.mp4
[*] Stopped recording.
[x] Exiting.


: 